In [0]:
from pyspark.sql import Row
from datetime import datetime
from pyspark.sql import functions as F

In [0]:
catalog =   "iics"
schema  =   "casemix"
table   =   "json_500mb_file_test"

In [0]:
file_path = "/Volumes/iics/casemix/history/json_test/sample_500_mb.json"
executed_by = spark.sql("SELECT current_user()").first()[0]

file_info = dbutils.fs.ls(file_path)[0]

json_df = spark.read.option("multiLine", "true").json(file_path)
file_structure = json_df.schema.json()

result_df = spark.createDataFrame([
    Row(
        file_name=file_info.name,
        file_path=file_info.path,
        file_size_mb=file_info.size / (1024.0 * 1024.0),
        file_modified=datetime.fromtimestamp(file_info.modificationTime / 1000),
        file_structure=file_structure,
        executed_by=executed_by
    )
]).withColumn(
    "file_structure", F.parse_json("file_structure")
).withColumn(
    "loaded_at", F.current_timestamp()
)

In [0]:
result_df.write.mode("overwrite").saveAsTable(
    f"{catalog}.{schema}.{table}"
)